In [ ]:
import shutil
from pathlib import Path

DR = Path('/content/drive/MyDrive/YOLO_sign_detection/dataset')
for split in ['train','val']:
    src = DR/ split / 'labels_yolo'
    dst = DR/ split / 'labels'
    # remove old raw files
    for old in dst.glob('*.txt'):
        old.unlink()
    # copy only your converted YOLO labels
    for f in src.glob('*.txt'):
        shutil.copy(f, dst/f.name)
print("✅ labels/ directories now contain only your YOLO‐formatted annotations.")


✅ labels/ directories now contain only your YOLO‐formatted annotations.


In [ ]:
import re
from pathlib import Path
from PIL import Image

def convert_and_preview(
    raw_lbl_dir: Path,
    img_dir: Path,
    yolo_lbl_dir: Path,
    split_name: str,
    preview: int = 5,
    class_id: int = 0
):
    """
    Reads raw 8‑point .txt files from raw_lbl_dir (which in your case IS dataset/.../labels),
    converts them to YOLO <class> cx cy w h format,
    writes outputs into yolo_lbl_dir, and prints the first few samples.
    """
    raw_lbl_dir   = Path(raw_lbl_dir)
    img_dir       = Path(img_dir)
    yolo_lbl_dir  = Path(yolo_lbl_dir)
    yolo_lbl_dir.mkdir(exist_ok=True)  # assume parent dirs exist

    def quad_to_yolo(verts, w, h):
        xs, ys     = verts[0::2], verts[1::2]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        cx = (x_min + x_max) / 2
        cy = (y_min + y_max) / 2
        bw = x_max - x_min
        bh = y_max - y_min
        return cx/w, cy/h, bw/w, bh/h

    samples = []
    converted = 0

    for raw_txt in sorted(raw_lbl_dir.glob("*.txt")):
        stem = raw_txt.stem
        img_path = img_dir / f"{stem}.jpg"
        if not img_path.exists():
            img_path = img_dir / f"{stem}.png"
        if not img_path.exists():
            continue  # no matching image

        iw, ih = Image.open(img_path).size
        raw_lines = raw_txt.read_text().splitlines()
        yolo_lines = []

        for line in raw_lines:
            parts = re.split(r"[, \t]+", line.strip())
            if len(parts) != 8:
                continue
            verts = list(map(float, parts))
            cx, cy, bw, bh = quad_to_yolo(verts, iw, ih)
            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if yolo_lines:
            (yolo_lbl_dir / f"{stem}.txt").write_text("\n".join(yolo_lines))
            converted += 1
            if len(samples) < preview:
                samples.append((stem, raw_lines, yolo_lines))

    print(f"\n[{split_name}] Converted {converted} files to YOLO format in {yolo_lbl_dir}")
    if samples:
        print(f"\nFirst {len(samples)} samples for {split_name}:")
        for stem, raw, yolo in samples:
            print(f"\nImage: {stem}")
            print("  Raw:", raw)
            print("  YOLO:", yolo)
    else:
        print(f"⚠️ No valid conversions for {split_name} — check your raw .txt files.")

# ─── Main ───
DRIVE_ROOT    = Path("/content/drive/MyDrive/YOLO_sign_detection")
# Your unconverted 8-point labels are currently in these folders:
TRAIN_RAW_LBL = DRIVE_ROOT / "dataset" / "train" / "labels"
VAL_RAW_LBL   = DRIVE_ROOT / "dataset" / "val"   / "labels"

# Where to write the new YOLO labels:
TRAIN_YOLO_LBL = DRIVE_ROOT / "dataset" / "train" / "labels_yolo"
VAL_YOLO_LBL   = DRIVE_ROOT / "dataset" / "val"   / "labels_yolo"

# Image folders:
TRAIN_IMG_DIR = DRIVE_ROOT / "dataset" / "train" / "images"
VAL_IMG_DIR   = DRIVE_ROOT / "dataset" / "val"   / "images"

# Run conversion + preview
convert_and_preview(TRAIN_RAW_LBL, TRAIN_IMG_DIR, TRAIN_YOLO_LBL, "train")
convert_and_preview(VAL_RAW_LBL,   VAL_IMG_DIR,   VAL_YOLO_LBL,   "val")


In [3]:
from pathlib import Path
from ultralytics import YOLO

DRIVE_ROOT = Path('/content/drive/MyDrive/YOLO_sign_detection')
DATA_DIR   = DRIVE_ROOT/'dataset'

# regenerate data.yaml
(Path('data.yaml')).write_text(f"""\
path: {DATA_DIR}
train: train/images
val:   val/images

names:
  0: sign
""")

# clear caches
for split in ['train','val']:
    c = DATA_DIR/split/'labels.cache'
    if c.exists(): c.unlink()

# train
model = YOLO('yolov8n.pt')
model.train(
    data='data.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    project=DRIVE_ROOT/'models',
    name='sign_detector'
)




Ultralytics 8.3.161 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=sign_detector21, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, pr

train: Scanning /content/drive/MyDrive/YOLO_sign_detection/dataset/train/labels... 535 images, 1215 backgrounds, 0 corrupt: 100%|██████████| 1750/1750 [00:17<00:00, 101.33it/s]


train: New cache created: /content/drive/MyDrive/YOLO_sign_detection/dataset/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 13.6±29.1 ms, read: 49.0±59.9 MB/s, size: 1232.9 KB)


val: Scanning /content/drive/MyDrive/YOLO_sign_detection/dataset/val/labels... 74 images, 176 backgrounds, 0 corrupt: 100%|██████████| 250/250 [00:02<00:00, 96.20it/s] 


val: New cache created: /content/drive/MyDrive/YOLO_sign_detection/dataset/val/labels.cache
Plotting labels to /content/drive/MyDrive/YOLO_sign_detection/models/sign_detector21/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/YOLO_sign_detection/models/sign_detector21
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        250        129      0.785      0.705      0.766      0.301



50 epochs completed in 0.692 hours.
Optimizer stripped from /content/drive/MyDrive/YOLO_sign_detection/models/sign_detector21/weights/last.pt, 6.2MB
Optimizer stripped from /content/drive/MyDrive/YOLO_sign_detection/models/sign_detector21/weights/best.pt, 6.2MB

Validating /content/drive/MyDrive/YOLO_sign_detection/models/sign_detector21/weights/best.pt...
Ultralytics 8.3.161 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:08<00:00,  1.07s/it]


                   all        250        129       0.84      0.776      0.819      0.324
Speed: 0.3ms preprocess, 2.6ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to /content/drive/MyDrive/YOLO_sign_detection/models/sign_detector21


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ab2cb092ed0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [4]:
# inference
model.predict(
    source=str(DATA_DIR/'val'/'images'),
    imgsz=640,
    conf=0.25,
    save=True,
    project=DRIVE_ROOT/'results',
    name='inference_on_val'
)


image 1/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1000).JPG: 480x640 (no detections), 9.5ms
image 2/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1001).JPG: 480x640 (no detections), 7.6ms
image 3/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1002).JPG: 480x640 (no detections), 18.7ms
image 4/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1003).JPG: 480x640 (no detections), 7.8ms
image 5/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1004).JPG: 480x640 (no detections), 8.7ms
image 6/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1005).JPG: 480x640 (no detections), 7.2ms
image 7/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1006).JPG: 480x640 (no detections), 8.0ms
image 8/250 /content/drive/MyDrive/YOLO_sign_detection/dataset/val/images/tsd (1007).JPG: 480x640 (no detections), 8.0ms
image 9/250 /content/drive/MyD

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'sign'}
 obb: None
 orig_img: array([[[163, 157, 144],
         [162, 156, 143],
         [163, 157, 144],
         ...,
         [204, 203, 189],
         [203, 202, 188],
         [206, 205, 191]],
 
        [[162, 156, 143],
         [163, 157, 144],
         [163, 157, 144],
         ...,
         [205, 204, 190],
         [205, 204, 190],
         [207, 206, 192]],
 
        [[163, 157, 144],
         [163, 157, 144],
         [162, 156, 143],
         ...,
         [206, 205, 191],
         [205, 204, 190],
         [206, 205, 191]],
 
        ...,
 
        [[112, 114, 108],
         [115, 117, 111],
         [115, 117, 111],
         ...,
         [  9,   9,   9],
         [ 11,  11,  11],
         [ 10,  10,  10]],
 
        [[112, 113, 109],
         [112, 113, 109],
         [115, 116, 112],
         ...,
         [  9,  10, 

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/YOLO_sign_detection'
except ModuleNotFoundError:
    print("Not in Colab – using local paths. Adjust DRIVE_ROOT if needed.")
    DRIVE_ROOT = './YOLO_sign_detection'
#


!nvidia-smi


Mounted at /content/drive
Tue Jul  1 17:39:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.6 MB/s eta 0:00:00


1750

In [7]:
import os
# ─── 1) FORCE CPU‑ONLY ─────────────────────────────────────────
os.environ['CUDA_VISIBLE_DEVICES'] = ''   # hide GPUs
import tensorflow as tf
print("GPUs visible to TF:", tf.config.list_physical_devices('GPU'))  # should print []

from pathlib import Path
from PIL import Image
import numpy as np
from tensorflow.keras import layers, models, backend as K

# ─── 2) PATH SETUP ──────────────────────────────────────────────
ROOT = Path('/content/drive/MyDrive/YOLO_sign_detection')

# Full‑image folders for classification:
TRAIN_IMG_DIR = ROOT/'classification_raw'/'train'/'images'
TEST_IMG_DIR  = ROOT/'classification_raw'/'test'/'images'

# Look for exactly one global label file under classification_raw:
def find_global_label(subdir):
    p = ROOT/'classification_raw'/subdir/'labels'
    txts = list(p.glob('*.txt'))
    if len(txts)==1:
        return txts[0]
    # fallback to detection dataset’s YOLO labels:
    alt = ROOT/'dataset'/subdir/'labels'
    alt_txts = list(alt.glob('*.txt'))
    if alt_txts:
        return alt_txts[0]
    return None

TRAIN_LABEL = find_global_label('train')
TEST_LABEL  = find_global_label('test')

if TRAIN_LABEL is None:
    raise FileNotFoundError("Couldn’t find any '.txt' under classification_raw/train/labels or dataset/train/labels")

print("Using TRAIN_LABEL:", TRAIN_LABEL)
print("Using TEST_LABEL: ", TEST_LABEL)

# ─── 3) PARSE GLOBAL LABELS ───────────────────────────────────
def parse_records(label_file, img_dir):
    recs = []
    for L in label_file.read_text().splitlines():
        parts = [p for p in L.strip().split(';') if p]
        if len(parts) < 8:
            continue
        img_name, cls_id = parts[0], int(parts[7])
        p = img_dir / img_name
        if p.exists():
            recs.append((p, cls_id))
    return recs

train_recs = parse_records(TRAIN_LABEL, TRAIN_IMG_DIR)
test_recs  = parse_records(TEST_LABEL,  TEST_IMG_DIR) if TEST_LABEL else []
assert train_recs, "No training records found—check your paths & label file!"

# ─── 4) LOAD IMAGES INTO NUMPY ────────────────────────────────
def load_numpy(recs, size=(128,128)):
    X, y = [], []
    for path, cls in recs:
        img = Image.open(path).convert('RGB').resize(size)
        X.append(np.array(img, dtype=np.float32)/255.0)
        y.append(cls)
    return np.stack(X, axis=0), np.array(y, dtype=np.int32)

X_train, y_train = load_numpy(train_recs)
X_test,  y_test  = load_numpy(test_recs) if test_recs else (None, None)

# ─── 5) ONE‑HOT ENCODE ─────────────────────────────────────────
num_classes = int(y_train.max()) + 1
y_train_oh  = tf.keras.utils.to_categorical(y_train, num_classes).astype('float32')
y_test_oh   = (tf.keras.utils.to_categorical(y_test, num_classes)
               .astype('float32')) if y_test is not None else None

# ─── 6) DEBUG PRINT ───────────────────────────────────────────
print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train_oh.shape, y_train_oh.dtype)
if X_test is not None:
    print("X_test:", X_test.shape, X_test.dtype)
    print("y_test:", y_test_oh.shape, y_test_oh.dtype)

# ─── 7) CLEAR & EAGER ─────────────────────────────────────────
K.clear_session()
tf.config.run_functions_eagerly(True)

# ─── 8) BUILD & COMPILE CNN ───────────────────────────────────
model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Conv2D(8, 3, activation='relu'), layers.MaxPool2D(),
    layers.Conv2D(16,3, activation='relu'), layers.MaxPool2D(),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(num_classes, activation='softmax'),
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    run_eagerly=True
)

# ─── 9) TRAIN & EVALUATE ──────────────────────────────────────
model.fit(
    X_train, y_train_oh,
    validation_data=(X_test, y_test_oh) if y_test_oh is not None else None,
    epochs=5, batch_size=8
)
if y_test_oh is not None:
    loss, acc = model.evaluate(X_test, y_test_oh, batch_size=8)
    print(f"✅ Test accuracy: {acc:.2%}")

# ─── 10) SAVE MODEL ───────────────────────────────────────────
model.save(ROOT/'final_full_image_classifier.h5')
print("Model saved to:", ROOT/'final_full_image_classifier.h5')


GPUs visible to TF: []


FileNotFoundError: Couldn’t find any '.txt' under classification_raw/train/labels or dataset/train/labels

GPUs visible to TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
